In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [19]:
df_gdp = read_table("""
    select * from sc_bronze.datagov_gdp
""")

df_gdp

,state,date,sector,RM(million),growth_yoy
0,Johor,2016-01-01,Agriculture,15029.966,-3.718
1,Johor,2016-01-01,Mining and Quarrying,569.326,19.697
2,Johor,2016-01-01,Manufacturing,34121.545,5.448
3,Johor,2016-01-01,Construction,8978.486,23.515
4,Johor,2016-01-01,Services,56266.132,6.046
...,...,...,...,...,...
670,Terengganu,2024-01-01,Agriculture,2941.000,3.900
671,Terengganu,2024-01-01,Mining and Quarrying,236.000,13.600
672,Terengganu,2024-01-01,Manufacturing,14628.000,3.900
673,Terengganu,2024-01-01,Construction,1370.000,17.500


In [7]:
df_state_pop = read_table("""
    select * from sc_bronze.datagov_population
""")

df_state_pop

,state,date,population
0,Johor,2016-01-01,3651800.0
1,Johor,2017-01-01,3697000.0
2,Johor,2018-01-01,3749400.0
3,Johor,2019-01-01,3761200.0
4,Johor,2020-01-01,4009700.0
...,...,...,...
155,W.P. Putrajaya,2021-01-01,115200.0
156,W.P. Putrajaya,2022-01-01,117000.0
157,W.P. Putrajaya,2023-01-01,118800.0
158,W.P. Putrajaya,2024-01-01,120300.0


## GDP per capita

In [21]:
# calculate state GDP total 2016-2025 and merge with population for gdp per capita
# adjust column names according to source table structure

df_gdp_state = (
    df_gdp
    .query('date >= "2016-01-01" and date <= "2025-12-31"')
    .groupby(['state', 'date'], as_index=False)
    .agg({'RM(million)': 'sum'})
    .assign(gdp=lambda d: d['RM(million)'] * 1_000_000)
)

df_pop_state = (
    df_state_pop
    .query('date >= "2016-01-01" and date <= "2025-12-31"')
    .groupby(['state', 'date'], as_index=False)
    .agg({'population': 'sum'})
)

# join and compute GDP per capita
state_gdp_per_capita = (
    df_gdp_state
    .merge(df_pop_state, on=['state', 'date'], how='inner')
    .assign(gdp_per_capita=lambda d: d['gdp'] / d['population'])
)

state_gdp_summary = (
    state_gdp_per_capita
    .drop(columns=['RM(million)'], errors='ignore')
)

state_gdp_summary

,state,date,gdp,population,gdp_per_capita
0,Johor,2016-01-01,1.149655e+11,3651800.0,31481.859631
1,Johor,2017-01-01,1.216971e+11,3697000.0,32917.802543
2,Johor,2018-01-01,1.289144e+11,3749400.0,34382.673761
3,Johor,2019-01-01,1.326171e+11,3761200.0,35259.255291
4,Johor,2020-01-01,1.267464e+11,4009700.0,31609.957104
...,...,...,...,...,...
112,Terengganu,2020-01-01,3.396334e+10,1149400.0,29548.759353
113,Terengganu,2021-01-01,3.521990e+10,1170700.0,30084.477663
114,Terengganu,2022-01-01,3.734553e+10,1186600.0,31472.723749
115,Terengganu,2023-01-01,3.818669e+10,1210000.0,31559.249587


In [22]:
df_underemp = read_table("""
    select * from sc_bronze.dosm_underemployment
""")

df_underemp

,year,age_group,underemp_type,qualification,underemp_graduate
0,2016,25 - 34,time,degree,6800.0
1,2016,25 - 34,time,diploma,6200.0
2,2016,25 - 34,skill,degree,132200.0
3,2016,25 - 34,skill,diploma,295600.0
4,2016,35 - 44,time,degree,3000.0
...,...,...,...,...,...
139,2024,≤ 24,skill,diploma,193900.0
140,2024,≥ 45,time,degree,7200.0
141,2024,≥ 45,time,diploma,6600.0
142,2024,≥ 45,skill,degree,94100.0
